# Finding out: can a small borrowed model learn to look things up instead of guessing?

The "Finding Out" paper's bet: a capable assistant doesn't need to know much; it needs to know how to find out.
Here a small pretrained model (SmolLM2-135M-Instruct) keeps its English, and we teach it only the loop:
notice a gap, search, note what it found, calculate exactly, ask the person, catch stale assumptions, check, answer.

Every episode happens in a freshly invented world (made-up towns, creatures, items, people), so nothing can be
memorised: the only way to be right is to find out. Seven task types are trained on; three are held out entirely.

**Conditions**
- `base`: SmolLM2-135M-Instruct, untrained, with two worked examples in the prompt
- `lora`: the same model with small LoRA adapters trained on the loop (the core weights stay frozen)
- `full`: the whole model fine-tuned (to compare how much English each approach costs)

**Measures** (the paper's tests): success; looking up when needed; *not* looking up when the answer was given
(honest gaps both ways); exact arithmetic through the calculator; asking when information is missing; catching a
stale assumption (with the harness's note switched on and off); unfamiliar task types never trained on; and
English kept, measured as perplexity on held-out SimpleStories.

**On Kaggle**: GPU T4, Internet on. About 1 to 1.5 hours. Results go to `FINDOUT_RESULTS.md`.

In [ ]:
import json, math, os, random, re, statistics as st, sys, time, subprocess
QUICK = True or "--quick" in sys.argv or os.environ.get("FINDOUT_QUICK") == "1"
try:
    import peft  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=False)
# Kaggle ships an old torchao that makes peft refuse to add LoRA layers; we don't use torchao, so tell peft it's absent.
try:
    import peft.import_utils as _piu
    _piu.is_torchao_available = lambda: False
    import peft.tuners.lora.torchao as _plt
    _plt.is_torchao_available = lambda: False
except Exception:
    pass
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

DEV = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
N_TRAIN = 600 if QUICK else 24000
N_TEST = 4 if QUICK else 60          # per task type
EPOCHS = 1 if QUICK else 2
BATCH, MAXLEN = 16, 640
T0 = time.time()


def log(*a):
    print(f"[{(time.time() - T0) / 60:5.1f} min]", *a, flush=True)

## The world: made-up facts the model can't know, and tools to find them out

Every episode gets a fresh random world (invented towns, creatures, items, people), so nothing can be
memorised: the only way to answer is to look things up. Tools:
- `<search>query</search>` returns sentences from the world's facts (with a distractor)
- `<calc>expression</calc>` does arithmetic exactly
- `<ask>question</ask>` goes to the person, who answers from what they know
The model also keeps a working picture with `<fact key="k">v</fact>` and `<assume key="k">v</assume>`.
When a fact replaces an assumption, the harness says which earlier searches rested on the old value.

In [ ]:
import json, random, re

SYL = ["ba", "lo", "rin", "ta", "vor", "zel", "mi", "qua", "dun", "pel", "sor", "ky", "neth", "ul", "gra",
       "fen", "osh", "tir", "wex", "ja", "bri", "mok", "sel", "um", "dra", "vin", "ko", "ral", "esh", "thu"]


def name(r, n=None):
    return "".join(r.choice(SYL) for _ in range(n or r.choice((2, 3))))


COLORS = ["blue", "red", "green", "grey", "golden", "black", "white", "purple", "orange", "brown"]
FOODS = ["moss", "beetles", "seeds", "river fish", "berries", "leaves", "mushrooms", "worms", "fruit", "grass"]
HABITATS = ["marshes", "caves", "pine forests", "deserts", "lakes", "mountains", "meadows", "reefs"]
REGIONS = ["the north coast", "the eastern hills", "the south valley", "the western plains", "the high lakes", "the central basin"]
MATERIALS = ["oak", "copper", "glass", "wool", "clay", "steel", "bamboo", "leather"]
JOBS = ["baker", "teacher", "nurse", "carpenter", "pilot", "farmer", "librarian", "mechanic"]


def make_world(r):
    W = {"creature": {}, "town": {}, "item": {}, "person": {}}
    used = set()

    def name(r, n=None):            # unique, and no name contains another (so search can't confuse them)
        while True:
            x = "".join(r.choice(SYL) for _ in range(n or r.choice((2, 3))))
            if not any(x in u or u in x for u in used):
                used.add(x)
                return x
    for _ in range(4):
        W["creature"][name(r)] = {"legs": r.choice([2, 4, 6, 8, 10]), "color": r.choice(COLORS),
                                  "diet": r.choice(FOODS), "habitat": r.choice(HABITATS)}
    for _ in range(5):
        W["town"][name(r).capitalize()] = {"population": r.randrange(800, 99000), "region": r.choice(REGIONS),
                                           "founded": r.randrange(1650, 1990)}
    for _ in range(4):
        W["item"][name(r)] = {"price": r.randrange(3, 95), "material": r.choice(MATERIALS),
                              "weight": r.randrange(1, 40)}
    for _ in range(4):
        W["person"][name(r).capitalize()] = {"age": r.randrange(19, 88), "job": r.choice(JOBS)}
    return W


def fact_sentence(kind, ent, attr, val):
    if kind == "creature":
        return {"legs": f"A {ent} has {val} legs.", "color": f"The {ent} is {val}.",
                "diet": f"The {ent} eats {val}.", "habitat": f"The {ent} lives in {val}."}[attr]
    if kind == "town":
        return {"population": f"{ent} has a population of {val}.", "region": f"{ent} is in {val}.",
                "founded": f"{ent} was founded in {val}."}[attr]
    if kind == "item":
        return {"price": f"A {ent} costs {val} coins.", "material": f"A {ent} is made of {val}.",
                "weight": f"A {ent} weighs {val} grams."}[attr]
    return {"age": f"{ent} is {val} years old.", "job": f"{ent} works as a {val}."}[attr]


def search(W, query):
    """Return facts about every entity named in the query (requested attribute first), else no results."""
    q = query.lower()
    hits = []
    for kind, ents in W.items():
        for ent, attrs in ents.items():
            if re.search(r"\b" + re.escape(ent.lower()) + r"s?\b", q):
                order = sorted(attrs, key=lambda a: (a not in q and not any(s in q for s in SYN.get(a, ())), a))
                hits += [fact_sentence(kind, ent, a, attrs[a]) for a in order[:2]]
    return " ".join(hits) if hits else "no results"


SYN = {"diet": ("eat", "food"), "legs": ("leg",), "habitat": ("live", "where"), "population": ("people", "big", "size"),
       "region": ("where", "area"), "founded": ("old", "when"), "price": ("cost", "much"), "weight": ("heavy", "weigh"),
       "material": ("made",), "age": ("old",), "job": ("work", "do")}


def calc(expr):
    if not re.fullmatch(r"[\d\s+\-*/().<>=]+", expr):
        return "error: only numbers and + - * / ( ) < > allowed"
    try:
        v = eval(expr, {"__builtins__": {}})
    except Exception as e:
        return f"error: {type(e).__name__}"
    if isinstance(v, bool):
        return "true" if v else "false"
    return str(int(v)) if float(v).is_integer() else f"{v:.2f}"

## Tasks

Each task builds a request, the person's hidden knowledge, a gold trajectory and a checker. `split`
says where it's used: "train" types are trained on; "test" types are never seen in training (unfamiliar
problems). Requests come in tidy and messy (spoken, lowercase, filler) styles; some messy phrasings are
held out for testing.

In [ ]:
MESSY_PRE = ["uhh ", "ok so ", "hey ", "so like ", "quick q ", "random but ", ""]
MESSY_POST = [" lol", " idk", "?", " thx", "", " if u know"]


def style(r, tidy, messy, split):
    if r.random() < 0.5:
        return tidy
    pre = r.choice(MESSY_PRE[:4] if split == "train" else MESSY_PRE[4:])
    return pre + messy + r.choice(MESSY_POST)


def S(kind, **kw):
    return dict(kind=kind, **kw)


def pick(r, W, kind):
    ent = r.choice(sorted(W[kind]))
    return ent, W[kind][ent]


def t_lookup(r, W, split):
    c, a = pick(r, W, "creature")
    attr = r.choice(["diet", "legs", "habitat", "color"])
    q = {"diet": ("What does a {c} eat?", "what do {c}s eat"), "legs": ("How many legs does a {c} have?", "how many legs does a {c} have"),
         "habitat": ("Where does the {c} live?", "where do {c}s live"), "color": ("What colour is a {c}?", "what color is a {c}")}[attr]
    req = style(r, q[0].format(c=c), q[1].format(c=c), split)
    val = a[attr]
    gold = [S("think", text=f"I don't know what a {c} is. Look it up."),
            S("search", text=f"{c} {attr}"),
            S("fact", key=f"{c}_{attr}", text=str(val)),
            S("answer", text=fact_sentence("creature", c, attr, val))]
    return dict(type="lookup", request=req, gold=gold, person={}, check=lambda ans, tr: str(val) in ans, needs_search=True)


def t_given(r, W, split):
    c, a = pick(r, W, "creature")
    attr = r.choice(["diet", "legs"])
    s = fact_sentence("creature", c, attr, a[attr])
    req = style(r, f"{s} So what does a {c} eat?" if attr == "diet" else f"{s} How many legs does a {c} have?",
                f"{s.lower()} so what does a {c} eat" if attr == "diet" else f"{s.lower()} how many legs does a {c} have", split)
    gold = [S("think", text="The answer is already in the message. No lookup needed."),
            S("answer", text=s)]
    return dict(type="given", request=req, gold=gold, person={}, check=lambda ans, tr: str(a[attr]) in ans, needs_search=False)


def t_calc(r, W, split):
    i, a = pick(r, W, "item")
    k = r.randrange(7, 99)
    known = r.random() < 0.5
    if known:
        req = style(r, f"A {i} costs {a['price']} coins. How much do {k} of them cost?", f"a {i} is {a['price']} coins, how much for {k}", split)
        gold = [S("think", text="Multiply exactly with the calculator.")]
    else:
        req = style(r, f"How much do {k} {i}s cost?", f"how much would {k} {i}s cost", split)
        gold = [S("think", text=f"I don't know the price of a {i}. Look it up, then multiply."),
                S("search", text=f"{i} price"), S("fact", key=f"{i}_price", text=str(a["price"]))]
    tot = a["price"] * k
    gold += [S("calc", text=f"{a['price']} * {k}"), S("answer", text=f"{k} {i}s cost {tot} coins.")]
    return dict(type="calc" if known else "lookup+calc", request=req, gold=gold, person={},
                check=lambda ans, tr: str(tot) in ans, needs_search=not known, needs_calc=True)


def t_compare(r, W, split):
    (t1, a1), (t2, a2) = [(t, W["town"][t]) for t in r.sample(sorted(W["town"]), 2)]
    req = style(r, f"Which is bigger, {t1} or {t2}?", f"is {t1.lower()} bigger than {t2.lower()}", split)
    big = t1 if a1["population"] > a2["population"] else t2
    gold = [S("think", text="I need both populations."),
            S("search", text=f"{t1} population"), S("fact", key=f"{t1}_population", text=str(a1["population"])),
            S("search", text=f"{t2} population"), S("fact", key=f"{t2}_population", text=str(a2["population"])),
            S("calc", text=f"{a1['population']} > {a2['population']}"),
            S("answer", text=f"{big} is bigger: {t1} has {a1['population']} people and {t2} has {a2['population']}.")]
    return dict(type="compare", request=req, gold=gold, person={}, check=lambda ans, tr: ans.strip().startswith(big) or f"{big} is bigger" in ans, needs_search=True)


def t_ask(r, W, split):
    t, a = pick(r, W, "town")
    req = style(r, "What region is my hometown in?", "what region is my hometown in", split)
    gold = [S("think", text="I don't know their hometown. Ask."),
            S("ask", text="Which town is your hometown?"),
            S("fact", key="hometown", text=t),
            S("search", text=f"{t} region"), S("fact", key=f"{t}_region", text=a["region"]),
            S("answer", text=f"{t} is in {a['region']}.")]
    return dict(type="ask", request=req, gold=gold, person={"town": t}, check=lambda ans, tr: a["region"] in ans,
                needs_search=True, needs_ask=True)


def t_stale(r, W, split):
    ta = r.choice(sorted(W["town"])); aa = W["town"][ta]
    others = [t for t in sorted(W["town"]) if W["town"][t]["region"] != aa["region"]]
    if not others:                      # every other town shares the region: move one
        others = [r.choice(sorted(set(W["town"]) - {ta}))]
        W["town"][others[0]]["region"] = r.choice([g for g in REGIONS if g != aa["region"]])
    tb = r.choice(others); ab = W["town"][tb]
    req = style(r, f"What region does my sister live in? I think she's in {ta}, but I'm not sure.",
                f"what region does my sister live in, i think shes in {ta.lower()} but not sure", split)
    gold = [S("assume", key="sister_town", text=ta),
            S("search", text=f"{ta} region"),
            S("reply_in", text=f"oh wait she moved to {tb} last year"),
            S("fact", key="sister_town", text=tb),
            S("harness_in"),
            S("think", text=f"The search for {ta} is stale now. Look up {tb}."),
            S("search", text=f"{tb} region"), S("fact", key=f"{tb}_region", text=ab["region"]),
            S("answer", text=f"She's in {tb}, which is in {ab['region']}.")]
    return dict(type="stale", request=req, gold=gold, person={"interrupt": f"oh wait she moved to {tb} last year"},
                check=lambda ans, tr: ab["region"] in ans and aa["region"] not in ans, needs_search=True)


def t_whose(r, W, split):
    (ta, aa), (tb, ab) = [(t, W["town"][t]) for t in r.sample(sorted(W["town"]), 2)]
    req = style(r, f"My brother lives in {ta} and I live in {tb}. How many people live in my brother's town?",
                f"my brother lives in {ta.lower()} and im in {tb.lower()}, how many people live in my brothers town", split)
    gold = [S("fact", key="brother_town", text=ta), S("fact", key="my_town", text=tb),
            S("search", text=f"{ta} population"), S("fact", key=f"{ta}_population", text=str(aa["population"])),
            S("check", text=f"The question is about the brother's town, {ta}, not {tb}."),
            S("answer", text=f"{ta}, your brother's town, has {aa['population']} people.")]
    return dict(type="whose", request=req, gold=gold, person={},
                check=lambda ans, tr: str(aa["population"]) in ans and str(ab["population"]) not in ans, needs_search=True)


# --- held-out task types (never trained on)
def u_cost_compare(r, W, split):
    (i1, a1), (i2, a2) = [(i, W["item"][i]) for i in r.sample(sorted(W["item"]), 2)]
    k1, k2 = r.randrange(3, 30), r.randrange(3, 30)
    c1, c2 = a1["price"] * k1, a2["price"] * k2
    while c1 == c2:
        k2 += 1; c2 = a2["price"] * k2
    more = f"{k1} {i1}s" if c1 > c2 else f"{k2} {i2}s"
    req = style(r, f"Do {k1} {i1}s cost more than {k2} {i2}s?", f"would {k1} {i1}s cost more than {k2} {i2}s", split)
    return dict(type="U:cost-compare", request=req, gold=[], person={},
                check=lambda ans, tr: (("yes" in ans.lower()[:12]) == (c1 > c2)) or more in ans, needs_search=True, needs_calc=True)


def u_three(r, W, split):
    ts = r.sample(sorted(W["town"]), 3)
    big = max(ts, key=lambda t: W["town"][t]["population"])
    req = style(r, f"Which is the biggest: {ts[0]}, {ts[1]} or {ts[2]}?", f"which is biggest out of {ts[0].lower()}, {ts[1].lower()} and {ts[2].lower()}", split)
    return dict(type="U:three-way", request=req, gold=[], person={}, check=lambda ans, tr: big in ans.split(".")[0], needs_search=True)


def u_person(r, W, split):
    p, a = pick(r, W, "person")
    req = style(r, f"How old is {p}?", f"how old is {p.lower()}", split)
    return dict(type="U:new-kind", request=req, gold=[], person={}, check=lambda ans, tr: str(a["age"]) in ans, needs_search=True)


TRAIN_TASKS = [t_lookup, t_given, t_calc, t_compare, t_ask, t_stale, t_whose]
TEST_ONLY = [u_cost_compare, u_three, u_person]

## Rendering an episode as text, and the harness that runs a model's actions

In [ ]:
SYSTEM = ("You figure things out. You know nothing about this world's towns, creatures, items or people, so look up what you need. "
          "Actions: <think>...</think> to reason; <search>query</search> to look something up; <calc>expression</calc> for arithmetic; "
          "<ask>question</ask> to ask the person; <fact key=\"k\">value</fact> to note something known; <assume key=\"k\">value</assume> to note "
          "a guess; <check>...</check> to check your answer against your notes; <answer>...</answer> to reply. "
          "If the answer is already in the message, just answer.")


class Harness:
    """Runs tools, keeps the working picture, and flags searches that rested on a replaced assumption."""
    def __init__(self, W, person):
        self.W, self.person, self.assumed, self.searches, self.log = W, person, {}, [], []
        self.interrupted = False

    def act(self, tag, attrs, body):
        """Return the observation text to insert after an action (or '' for none)."""
        self.log.append((tag, attrs, body))
        if tag == "search":
            self.searches.append(body)
            res = f"<result>{search(self.W, body)}</result>"
            if self.person.get("interrupt") and not self.interrupted:
                self.interrupted = True
                res += f"\n<reply>{self.person['interrupt']}</reply>"
            return res
        if tag == "calc":
            return f"<result>{calc(body)}</result>"
        if tag == "ask":
            t = self.person.get("town")
            return f"<reply>{t if t else 'not sure, sorry'}</reply>"
        if tag == "assume":
            self.assumed[attrs.get("key")] = body
            return ""
        if tag == "fact":
            k = attrs.get("key")
            old = self.assumed.pop(k, None)
            if old is not None and old != body:
                stale = [s for s in self.searches if old.lower() in s.lower()]
                return f"<harness>{k} changed: assumed {old}, now {body}. Rested on {old}: {', '.join(repr(s) for s in stale) or 'nothing'}.</harness>"
            return ""
        return ""


ACTION_RE = re.compile(r"<(think|search|calc|ask|fact|assume|check|answer)((?:\s+\w+=\"[^\"]*\")*)>(.*?)</\1>", re.S)
TOOL_ENDS = ("</search>", "</calc>", "</ask>", "</fact>", "</assume>", "</answer>")


def attrs_of(s):
    return dict(re.findall(r"(\w+)=\"([^\"]*)\"", s))


def render_gold(task, W):
    """Gold trajectory -> (text, mask) with observations produced by the real harness (so train and test match)."""
    H = Harness(W, task["person"])
    parts = []
    for st in task["gold"]:
        k = st["kind"]
        if k in ("reply_in", "harness_in"):
            continue                     # these arrive from the harness itself
        tag = f'<{k} key="{st["key"]}">{st["text"]}</{k}>' if k in ("fact", "assume") else f"<{k}>{st['text']}</{k}>"
        parts.append((tag, True))
        if k == "answer":
            break
        obs = H.act(k, {"key": st.get("key")} if "key" in st else {}, st["text"])
        parts.append(("\n" + obs + "\n" if obs else "\n", False))   # separators come from the harness, as in run_model
    return parts


def run_model(gen, task, W, max_steps=14, harness_on=True):
    """gen(prefix_text) -> next chunk ending at a closing tag. Returns (answer, harness)."""
    H = Harness(W, task["person"])
    text = ""
    for _ in range(max_steps):
        chunk = gen(text)
        if not chunk:
            break
        text += chunk
        m = list(ACTION_RE.finditer(chunk))
        if not m:
            break
        a = m[-1]
        tag, at, body = a.group(1), attrs_of(a.group(2)), a.group(3).strip()
        if tag == "answer":
            return body, H, text
        obs = H.act(tag, at, body)
        if obs.startswith("<harness>") and not harness_on:
            obs = ""
        text += "\n" + obs + "\n" if obs else "\n"
    return "", H, text


def score(task, answer, H):
    used = {t for t, _, _ in H.log}
    return {"type": task["type"], "ok": bool(answer) and bool(task["check"](answer, H)),
            "searched": "search" in used, "needs_search": task.get("needs_search", False),
            "calc": "calc" in used, "needs_calc": task.get("needs_calc", False),
            "asked": "ask" in used, "needs_ask": task.get("needs_ask", False), "steps": len(H.log)}


def make_split(seed, n, split, tasks):
    r = random.Random(seed)
    out = []
    for _ in range(n):
        W = make_world(r)
        out.append((r.choice(tasks)(r, W, split), W))
    return out

## Data

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
END = "<|im_end|>"


def prompt_text(request, shots=()):
    msgs = [{"role": "system", "content": SYSTEM}]
    for q, a in shots:
        msgs += [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
    msgs.append({"role": "user", "content": request})
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def encode_episode(task, W):
    """Full text with a character mask (1 = the model's own actions), tokenised with offsets."""
    p = prompt_text(task["request"])
    text, mask = p, [0] * len(p)
    for piece, trained in render_gold(task, W):
        text += piece; mask += [int(trained)] * len(piece)
    text += END; mask += [1] * len(END)
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True, truncation=True, max_length=MAXLEN)
    lab = [i if mask[s] else -100 for i, (s, e) in zip(enc["input_ids"], enc["offset_mapping"])]
    return enc["input_ids"], lab


log("building data")
train = make_split(1, N_TRAIN, "train", TRAIN_TASKS)
train_enc = [encode_episode(t, W) for t, W in train]
lens = [len(i) for i, _ in train_enc]
log(f"{len(train_enc)} training episodes, mean {st.mean(lens):.0f} tokens, max {max(lens)}")

TESTS = {}
for i, f in enumerate(TRAIN_TASKS + TEST_ONLY):
    TESTS[f.__name__] = make_split(100 + i, N_TEST, "test", [f])

SHOTS = []
for f in (t_lookup, t_stale):                    # two worked examples for the untrained baseline
    t, W = make_split(7, 1, "train", [f])[0]
    SHOTS.append((t["request"], "".join(p for p, _ in render_gold(t, W))))

## Training

In [ ]:
def load(kind):
    m = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to(DEV)
    if kind == "lora":
        from peft import LoraConfig, get_peft_model
        m = get_peft_model(m, LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, task_type="CAUSAL_LM",
                                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    return m


def collate(batch):
    L = max(len(i) for i, _ in batch)
    ids = torch.full((len(batch), L), tok.pad_token_id)
    lab = torch.full((len(batch), L), -100)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for b, (i, l) in enumerate(batch):
        ids[b, :len(i)] = torch.tensor(i); lab[b, :len(l)] = torch.tensor(l); att[b, :len(i)] = 1
    return ids.to(DEV), lab.to(DEV), att.to(DEV)


def train_model(kind):
    m = load(kind)
    lr = 2e-4 if kind == "lora" else 5e-5
    params = [p for p in m.parameters() if p.requires_grad]
    log(f"{kind}: training {sum(p.numel() for p in params)/1e6:.2f}M of {sum(p.numel() for p in m.parameters())/1e6:.1f}M parameters")
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=DEV == "cuda")
    data = sorted(train_enc, key=lambda x: len(x[0]))           # length-bucketed batches, shuffled order
    batches = [data[i:i + BATCH] for i in range(0, len(data), BATCH)]
    total = EPOCHS * len(batches)
    step = 0
    m.train()
    for ep in range(EPOCHS):
        random.Random(ep).shuffle(batches)
        for b in batches:
            frac = step / total
            for g in opt.param_groups:
                g["lr"] = lr * min(1, (step + 1) / 50) * 0.5 * (1 + math.cos(math.pi * frac))
            ids, lab, att = collate(b)
            with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
                out = m(input_ids=ids, attention_mask=att)
            loss = F.cross_entropy(out.logits[:, :-1].float().reshape(-1, out.logits.shape[-1]), lab[:, 1:].reshape(-1), ignore_index=-100)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update()
            step += 1
            if step % 200 == 0 or step == total:
                log(f"  {kind} step {step}/{total} loss {loss.item():.3f}")
    m.eval()
    return m

## Running a model in the world (batched)

In [ ]:
CLOSE = re.compile(r"</(think|search|calc|ask|fact|assume|check|answer)>")


@torch.no_grad()
def run_batch(m, items, shots=(), harness_on=True, max_steps=14, new_tokens=72):
    """items: [(task, W)]. Returns [(answer, Harness, text)] in the same order."""
    st_ = [{"task": t, "H": Harness(W, t["person"]), "text": "", "ans": None, "done": False,
            "prompt": prompt_text(t["request"], shots)} for t, W in items]
    for _ in range(max_steps):
        live = [s for s in st_ if not s["done"]]
        if not live:
            break
        for i in range(0, len(live), 32):
            chunk = live[i:i + 32]
            enc = tok([s["prompt"] + s["text"] for s in chunk], return_tensors="pt", padding=True, add_special_tokens=False).to(DEV)
            with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
                out = m.generate(**enc, max_new_tokens=new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
            for s, seq in zip(chunk, out[:, enc["input_ids"].shape[1]:]):
                gen = tok.decode(seq, skip_special_tokens=True)
                c = CLOSE.search(gen)
                if not c:
                    s["done"] = True; continue
                piece = gen[:c.end()]
                s["text"] += piece
                a = list(ACTION_RE.finditer(piece))
                if not a:
                    s["done"] = True; continue
                tag, at, body = a[-1].group(1), attrs_of(a[-1].group(2)), a[-1].group(3).strip()
                if tag == "answer":
                    s["ans"], s["done"] = body, True; continue
                obs = s["H"].act(tag, at, body)
                if obs.startswith("<harness>") and not harness_on:
                    obs = ""
                s["text"] += "\n" + obs + "\n" if obs else "\n"
    return [(s["ans"] or "", s["H"], s["text"]) for s in st_]


def evaluate(m, name, shots=(), harness_on=True, only=None):
    res = {}
    for tname, items in TESTS.items():
        if only and tname not in only:
            continue
        outs = run_batch(m, items, shots, harness_on)
        res[tname] = [score(t, a, H) for (t, _), (a, H, _) in zip(items, outs)]
        res[tname + "_example"] = outs[0][2][-600:]
        ok = st.mean(r["ok"] for r in res[tname])
        log(f"  {name:14} {tname:15} success {ok*100:5.1f}%")
    return res

## English kept: perplexity on held-out simple stories

In [ ]:
def load_english(n=300):
    try:
        from datasets import load_dataset
        ds = load_dataset("SimpleStories/SimpleStories", split="test", streaming=True)
        return [r["story"] for _, r in zip(range(n), ds)]
    except Exception as e:
        log(f"could not load SimpleStories ({e}); skipping the English check")
        return []


ENGLISH = load_english(20 if QUICK else 300)


@torch.no_grad()
def perplexity(m):
    if not ENGLISH:
        return None
    tot, cnt = 0.0, 0
    for i in range(0, len(ENGLISH), 16):
        enc = tok(ENGLISH[i:i + 16], return_tensors="pt", padding=True, truncation=True, max_length=256).to(DEV)
        lab = enc["input_ids"].masked_fill(enc["attention_mask"] == 0, -100)
        with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
            lg = m(**enc).logits
        l = F.cross_entropy(lg[:, :-1].float().reshape(-1, lg.shape[-1]), lab[:, 1:].reshape(-1), ignore_index=-100, reduction="sum")
        tot += l.item(); cnt += (lab[:, 1:] != -100).sum().item()
    return math.exp(tot / cnt)

## Run everything

In [ ]:
def summarise(R):
    pct = lambda v: "–" if v is None else f"{v*100:.0f}%"
    rows = ["| task | " + " | ".join(R) + " |", "|---" * (len(R) + 1) + "|"]
    names = [n for n in next(iter(R.values()))["eval"] if not n.endswith("_example")]
    for n in names:
        rows.append(f"| {n.replace('t_', '').replace('u_', 'unfamiliar: ')} | " + " | ".join(pct(st.mean(x['ok'] for x in R[c]['eval'][n])) for c in R) + " |")
    L = ["# Finding out: results", "", f"SmolLM2-135M-Instruct, {N_TRAIN} training episodes, {N_TEST} test episodes per task.", "",
         "## Success by task", ""] + rows + ["", "## The loop itself", "",
         "| measure | " + " | ".join(R) + " |", "|---" * (len(R) + 1) + "|"]

    def frac(c, names_, cond, val):
        xs = [x for n in names_ for x in R[c]["eval"].get(n, []) if cond(x)]
        return st.mean(val(x) for x in xs) if xs else None
    need = [n for n in names if n != "t_given"]
    L.append("| looked up when it needed to | " + " | ".join(pct(frac(c, need, lambda x: x["needs_search"], lambda x: x["searched"])) for c in R) + " |")
    L.append("| looked up when the answer was given (lower is better) | " + " | ".join(pct(frac(c, ["t_given"], lambda x: True, lambda x: x["searched"])) for c in R) + " |")
    L.append("| used the calculator for arithmetic | " + " | ".join(pct(frac(c, names, lambda x: x["needs_calc"], lambda x: x["calc"])) for c in R) + " |")
    L.append("| asked when information was missing | " + " | ".join(pct(frac(c, ["t_ask"], lambda x: True, lambda x: x["asked"])) for c in R) + " |")
    L.append("| English perplexity (lower is better) | " + " | ".join("–" if R[c]["ppl"] is None else f"{R[c]['ppl']:.1f}" for c in R) + " |")
    if any("stale_off" in R[c] for c in R):
        L += ["", "## Stale assumptions with the harness note switched off", ""]
        for c in R:
            if "stale_off" in R[c]:
                on = st.mean(x["ok"] for x in R[c]["eval"]["t_stale"]); off = st.mean(x["ok"] for x in R[c]["stale_off"]["t_stale"])
                L.append(f"- {c}: {on*100:.0f}% with the note, {off*100:.0f}% without")
    L += ["", "## One example per task (end of the trajectory, for the trained LoRA model)", ""]
    best = "lora" if "lora" in R else list(R)[-1]
    for n in names:
        L += [f"**{n}**", "```", R[best]["eval"][n + "_example"], "```", ""]
    return "\n".join(L) + "\n"


def main():
    R = {}
    base = load("base"); base.eval()
    log("evaluating base (2 worked examples in the prompt)")
    R["base"] = {"eval": evaluate(base, "base", shots=SHOTS), "ppl": perplexity(base)}
    del base; torch.cuda.empty_cache() if DEV == "cuda" else None
    for kind in ("lora", "full"):
        m = train_model(kind)
        log(f"evaluating {kind}")
        R[kind] = {"eval": evaluate(m, kind), "ppl": perplexity(m)}
        R[kind]["stale_off"] = evaluate(m, kind + " no-harness", harness_on=False, only=["t_stale"])
        if kind == "lora":
            os.makedirs("findout_lora", exist_ok=True)
            merged = m.merge_and_unload()
            merged.save_pretrained("findout_lora/merged"); tok.save_pretrained("findout_lora/merged")
            del merged
        del m; torch.cuda.empty_cache() if DEV == "cuda" else None
        json.dump({c: {k: v for k, v in R[c].items()} for c in R}, open("findout_results.json", "w"), indent=1, default=str)
    text = summarise(R)
    open("FINDOUT_RESULTS.md", "w").write(text)
    print(text)
    export_gguf()


def export_gguf():
    """Optional: convert the merged LoRA model to GGUF for Ollama with llama.cpp's converter."""
    try:
        if not os.path.exists("llama.cpp"):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/ggml-org/llama.cpp"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gguf"], check=False)
        r = subprocess.run([sys.executable, "llama.cpp/convert_hf_to_gguf.py", "findout_lora/merged", "--outtype", "f16",
                            "--outfile", "findout.gguf"], capture_output=True, text=True)
        log("gguf:", "ok" if r.returncode == 0 else r.stderr[-800:])
        open("Modelfile", "w").write("FROM ./findout.gguf\nPARAMETER temperature 0\nPARAMETER num_ctx 2048\nPARAMETER stop </search>\nPARAMETER stop </calc>\nPARAMETER stop </ask>\nPARAMETER stop </answer>\n")
    except Exception as e:
        log(f"gguf export skipped: {e}")

In [ ]:
if __name__ == "__main__" or "ipykernel" in sys.modules:
    main()